In [151]:
import polars as pl
import numpy as np
from bidict import bidict

from typing import Sequence

from summer3.polarized.properties import Property, PropertyTable, LazyExpr

In [36]:
age = Property("age", ["infant", "child","adult", "older"])
state = Property("state", ["S", "I", "R"])
severity = Property("severity", ["mild", "severe"])

In [55]:
class ProxyProperty(Property):
    def __init__(self, existing, prepend):
        self._prepender = prepend
        self._existing = existing
        super().__init__(f"{prepend}({existing.name})", existing.traits)

    def __hash__(self):
        return hash((self._prepender, hash(self._existing)))

In [56]:
def Source(prop):
    return ProxyProperty(prop, "source")

def Dest(prop):
    return ProxyProperty(prop, "dest")

In [37]:
pt = PropertyTable.from_property(state).stratify(severity, state=="I").stratify(age, state)

In [150]:
from numpy.typing import NDArray

In [166]:
class CategoryGroup:
    def __init__(self, queries: Sequence[LazyExpr]):
        self.queries = np.array(queries)

    def __getitem__(self, idx):
        if isinstance(idx, int):
            idx = np.s_[idx:idx+1]
        return CategoryGroup(self.queries[idx])
    
    def __repr__(self):
        return f"CategoryGroup[{self.queries}]"

In [138]:
def valid_columns(df):
    validity_s = df.select(~pl.all().is_null().all())
    valid_columns = [c.name for c in validity_s if c[0] and (c.name != "index")]
    return valid_columns

def invalid_columns(df):
    invalidity_s = df.select(pl.all().is_null().all())
    invalid_columns = [c.name for c in invalidity_s if c[0]]
    return invalid_columns

def cat_counts(df, column):
    return df.group_by(column).len()#.rename({"len": "repetition_count"})

def repeat_on_other(df1, df2, column):
    # 1. Calculate value_counts for the 'category' column
    category_counts2 = df2.group_by(column).len().rename({"len": "repetition_count"})

    # 2. Join with the original DataFrame
    df1_with_counts = df1.join(category_counts2, on=column, how="left")

    # 3. Apply repeat_by and flatten
    repeated_df = df1_with_counts.select(
        pl.all().repeat_by(pl.col("repetition_count")).flatten()
    ).drop("repetition_count") # Drop the repetition_count column if not needed

    return repeated_df

def relabel_props(df, prepend):
    renamed = df.rename({col:f"{prepend}__{col}" for col in df.columns})
    return renamed

def create_index(df):
    return df.with_columns(index=np.arange(len(df)))

def expr_from_dict(prop_dict):
    expr = None
    for col,val in prop_dict.items():
        if expr is None:
            expr = pl.col(col) == val
        else:
            expr = expr & (pl.col(col) == val)
    return expr

def repeat_from_counts(fs, basis_columns, ref_src=True, basis_mappings=None):
    if basis_mappings is None:
        basis_mappings = {}

    if ref_src:
        ref = fs.src_df
        target = fs.dest_df
        extras = fs.created_props
    else:
        ref = fs.dest_df
        target = fs.src_df
        extras = fs.erased_props
        basis_mappings = {k:v.inv for k,v in basis_mappings.items()}

    cc = target.group_by(basis_columns + extras).len().group_by(basis_columns).len()

    print(cc)

    accum = []
    for row in cc.iter_rows(named=True):
        length = row.pop("len")

        print(row)
        row = row | {k:mapping[row[k]] for k,mapping in basis_mappings.items()}

        #expr = None
        #for col,val in row.items():
        #    if expr is None:
        #        expr = pl.col(col) == val
        #    else:
        #        expr = expr & (pl.col(col) == val)

        expr = expr_from_dict(row)

        basis_df = ref.filter(expr)
        adf_exp = pl.concat([basis_df]*length)
        accum.append(adf_exp)

    return pl.concat(accum)

def perfect_tiling(df: pl.DataFrame, ref_col: str) -> bool:
    """Check if a DataFrame contains only repeated blocks, that tile exactly
    an integer number of times over the 'axis' specified by ref_col, such
    that the 1d array represented by this table could be represented as 2d
    by unwrapping over this column

    Args:
        df: The DataFrame to check
        ref_col: _description_

    Returns:
        Tiling status
    """
    
    uvals = df.select(pl.col(ref_col)).unique().to_numpy()[:,0]
    ref_block = df.filter(pl.col(ref_col) == uvals[0])
    ref_block = ref_block.drop(invalid_columns(ref_block) + ["index", ref_col])

    a = ref_block
    b = df.drop(invalid_columns(df) + ["index", ref_col])

    if set(a.columns) != set(b.columns):
        return False

    comp_tiled = pl.DataFrame(np.tile(a.to_numpy().T,len(b)//len(a)).T, schema=a.columns)

    return (comp_tiled==b).select(pl.all_horizontal(pl.all()).all()).item()

In [ ]:
class FlowSpec:
    def __init__(self, srcq: LazyExpr|CategoryGroup, destq: LazyExpr|CategoryGroup, pt: PropertyTable):
        self.srcq = srcq
        self.destq = destq
        self.pt = pt

        if isinstance(srcq, CategoryGroup):
            if isinstance(destq, CategoryGroup):
                #+++
                #IMPLEMENT ME!
                pass

        self.src_table = pt.filter(srcq)
        self.dest_table = pt.filter(destq)

        self.src_df = self.src_table.df
        self.dest_df = self.dest_table.df

        self.src_props = src_props = valid_columns(self.src_df)
        self.dest_props = dest_props = valid_columns(self.dest_df)

        self.common_props = list(set(src_props).intersection(set(dest_props)))

        self.erased_props = list(set(src_props).difference(set(dest_props)))
        self.created_props = list(set(dest_props).difference(set(src_props)))

    def get_policy_tables(self):
        props_matched = [] # A->A
        props_moved = [] # Nothing in common (moved)
        props_to_map = []

        for column in self.common_props:
            src_props = self.src_df[column].unique()
            dest_props = self.dest_df[column].unique()

            if (src_props == dest_props).all():
                props_matched.append(column)
            elif (src_props != dest_props).all():
                props_moved.append(column)
            else:
                props_to_map.append(column)        

        return {"matched": props_matched, "moved": props_moved, "mapped": props_to_map}
    
    def realise_mapping(self, basis_mappings=None):

        policy = self.get_policy_tables()

        #if len(policy["matched"]) > 1:
        #    raise Exception("Only single match supported for now")
        # policy["matched"] +++
        basis = policy["matched"] + policy["moved"]
        src_mapped = create_index(relabel_props(repeat_from_counts(self, basis, True, basis_mappings), "source"))
        dest_mapped = create_index(relabel_props(repeat_from_counts(self, basis, False, basis_mappings), "dest"))

        joined = src_mapped.join(dest_mapped, on="index")

        return joined
        

In [140]:
pension = Property("pension", ["no", "yes"])
pt_pens = pt.stratify(pension, age=="older")

In [141]:
infection = FlowSpec(state == "S", state == "I", pt)
ageing = FlowSpec(age < "older", age > "infant", pt_pens)

In [142]:
ageing.get_policy_tables()

{'matched': ['severity_0', 'state_0'], 'moved': ['age_0'], 'mapped': []}

In [144]:
ageing.realise_mapping({"age_0": bidict({0: 1, 1:2, 2:3}).inverse}).sort("source__index")

shape: (12, 4)
┌────────────┬─────────┬───────┬─────┐
│ severity_0 ┆ state_0 ┆ age_0 ┆ len │
│ ---        ┆ ---     ┆ ---   ┆ --- │
│ i64        ┆ i64     ┆ i64   ┆ u32 │
╞════════════╪═════════╪═══════╪═════╡
│ null       ┆ 2       ┆ 2     ┆ 1   │
│ 1          ┆ 1       ┆ 1     ┆ 1   │
│ 0          ┆ 1       ┆ 3     ┆ 2   │
│ null       ┆ 0       ┆ 2     ┆ 1   │
│ null       ┆ 2       ┆ 1     ┆ 1   │
│ …          ┆ …       ┆ …     ┆ …   │
│ 0          ┆ 1       ┆ 2     ┆ 1   │
│ 1          ┆ 1       ┆ 3     ┆ 2   │
│ null       ┆ 2       ┆ 3     ┆ 2   │
│ 0          ┆ 1       ┆ 1     ┆ 1   │
│ null       ┆ 0       ┆ 1     ┆ 1   │
└────────────┴─────────┴───────┴─────┘
{'severity_0': None, 'state_0': 2, 'age_0': 2}
{'severity_0': 1, 'state_0': 1, 'age_0': 1}
{'severity_0': 0, 'state_0': 1, 'age_0': 3}
{'severity_0': None, 'state_0': 0, 'age_0': 2}
{'severity_0': None, 'state_0': 2, 'age_0': 1}
{'severity_0': 1, 'state_0': 1, 'age_0': 2}
{'severity_0': None, 'state_0': 0, 'age_0': 3}
{'

C:\Users\dshi0012\AppData\Local\Temp\ipykernel_20624\1957561110.py:39: UserWarning: Comparisons with None always result in null. Consider using `.is_null()` or `.is_not_null()`.
  expr = pl.col(col) == val


source__state_0,source__severity_0,source__age_0,source__pension_0,source__index,index,dest__state_0,dest__severity_0,dest__age_0,dest__pension_0,dest__index
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,0,0,null,3,7,1,1,1,null,8
1,0,1,null,4,4,1,0,1,null,4
1,0,2,null,5,1,1,1,2,null,9
1,0,2,null,5,2,1,0,3,0,6
1,1,0,null,6,0,1,0,2,null,5
1,1,1,null,7,3,1,0,3,1,7
1,1,2,null,8,5,1,1,3,0,10
1,1,2,null,8,6,1,1,3,1,11


In [68]:
ageing.common_props

['severity_0', 'age_0', 'state_0']

In [71]:
ageing.common_props

['severity_0', 'age_0', 'state_0']

In [ ]:
# What if...
# For each prop, we determine a mapping;
# either directly from the queries (1:1, 1:many, many:1) or from user-supplied mappings


In [98]:
ageing.src_df.with_columns(age_0 = pl.col("age_0") + 1)

state_0,severity_0,age_0,pension_0,index
i64,i64,i64,i64,i64
0,null,1,null,0
0,null,2,null,1
0,null,3,null,2
1,0,1,null,3
1,0,2,null,4
…,…,…,…,…
1,1,2,null,7
1,1,3,null,8
2,null,1,null,9


In [ ]:
ageing.src_with_columns(age_0 = pl.col("age_0") + 1), ["age_0"], True)

AttributeError: 'FlowSpec' object has no attribute 'with_columns'

In [78]:
ageing.created_props

['pension_0']

In [79]:
ageing.src_df

state_0,severity_0,age_0,pension_0,index
i64,i64,i64,i64,i64
0,null,0,null,0
0,null,1,null,1
0,null,2,null,2
1,0,0,null,3
1,0,1,null,4
…,…,…,…,…
1,1,1,null,7
1,1,2,null,8
2,null,0,null,9


In [89]:
ddf = ageing.src_df

ddf.unique(["state_0","age_0","severity_0"])

state_0,severity_0,age_0,pension_0,index
i64,i64,i64,i64,i64
1,0,0,null,3
2,null,1,null,10
1,0,1,null,4
1,1,2,null,8
2,null,0,null,9
…,…,…,…,…
0,null,2,null,2
2,null,2,null,11
1,1,0,null,6


In [88]:
ddf = ageing.dest_df

ddf.unique(["state_0","age_0","severity_0"])

state_0,severity_0,age_0,pension_0,index
i64,i64,i64,i64,i64
1,0,3,0,6
1,1,1,null,8
2,null,1,null,12
1,0,1,null,4
0,null,1,null,0
…,…,…,…,…
1,1,2,null,9
1,1,3,0,10
0,null,2,null,1


state_0,severity_0,age_0,pension_0,index
i64,i64,i64,i64,i64
0,null,1,null,0
0,null,2,null,1
0,null,3,0,2
0,null,3,1,3
1,0,1,null,4
…,…,…,…,…
1,1,3,1,11
2,null,1,null,12
2,null,2,null,13


In [ ]:
repeat_from_counts(ageing, ["state_0", "severity_0"])

C:\Users\dshi0012\AppData\Local\Temp\ipykernel_20624\2715661282.py:41: UserWarning: Comparisons with None always result in null. Consider using `.is_null()` or `.is_not_null()`.
  expr = expr & (pl.col(col) == val)


state_0,severity_0,age_0,pension_0,index
i64,i64,i64,i64,i64
1,0,0,null,3
1,0,1,null,4
1,0,2,null,5
1,0,0,null,3
1,0,1,null,4
…,…,…,…,…
1,1,1,null,7
1,1,2,null,8
1,1,0,null,6


In [70]:
ageing.realise_mapping()

C:\Users\dshi0012\AppData\Local\Temp\ipykernel_20624\2715661282.py:39: UserWarning: Comparisons with None always result in null. Consider using `.is_null()` or `.is_not_null()`.
  expr = pl.col(col) == val


source__state_0,source__severity_0,source__age_0,source__pension_0,source__index,index,dest__state_0,dest__severity_0,dest__age_0,dest__pension_0,dest__index
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,1,0,null,6,0,1,1,1,null,8
1,1,1,null,7,1,1,1,2,null,9
1,1,2,null,8,2,1,1,3,0,10
1,1,0,null,6,3,1,1,3,1,11
1,1,1,null,7,4,1,0,1,null,4
1,1,2,null,8,5,1,0,2,null,5
1,1,0,null,6,6,1,0,3,0,6
1,1,1,null,7,7,1,0,3,1,7


In [66]:
pt_pens

PropertyTable
shape: (20, 5)
┌─────────┬────────────┬────────┬───────────┬───────┐
│ state_0 ┆ severity_0 ┆ age_0  ┆ pension_0 ┆ index │
│ ---     ┆ ---        ┆ ---    ┆ ---       ┆ ---   │
│ str     ┆ str        ┆ str    ┆ str       ┆ i64   │
╞═════════╪════════════╪════════╪═══════════╪═══════╡
│ S       ┆ null       ┆ infant ┆ null      ┆ 0     │
│ S       ┆ null       ┆ child  ┆ null      ┆ 1     │
│ S       ┆ null       ┆ adult  ┆ null      ┆ 2     │
│ S       ┆ null       ┆ older  ┆ no        ┆ 3     │
│ S       ┆ null       ┆ older  ┆ yes       ┆ 4     │
│ …       ┆ …          ┆ …      ┆ …         ┆ …     │
│ R       ┆ null       ┆ infant ┆ null      ┆ 15    │
│ R       ┆ null       ┆ child  ┆ null      ┆ 16    │
│ R       ┆ null       ┆ adult  ┆ null      ┆ 17    │
│ R       ┆ null       ┆ older  ┆ no        ┆ 18    │
│ R       ┆ null       ┆ older  ┆ yes       ┆ 19    │
└─────────┴────────────┴────────┴───────────┴───────┘

In [65]:
ageing.get_policy_tables()

{'matched': ['severity_0', 'state_0'], 'moved': ['age_0'], 'mapped': []}

In [58]:
infection.realise_mapping().sort("source__index")

source__state_0,source__severity_0,source__age_0,source__index,index,dest__state_0,dest__severity_0,dest__age_0,dest__index
i64,i64,i64,i64,i64,i64,i64,i64,i64
0,null,0,0,6,1,0,1,1
0,null,0,0,7,1,1,1,5
0,null,1,1,0,1,0,2,2
0,null,1,1,1,1,1,2,6
0,null,2,2,2,1,0,3,3
0,null,2,2,3,1,1,3,7
0,null,3,3,4,1,0,0,0
0,null,3,3,5,1,1,0,4


In [23]:
infection.realise_mapping()

source__state_0,source__severity_0,source__age_0,source__index,index,dest__state_0,dest__severity_0,dest__age_0,dest__index
i64,i64,i64,i64,i64,i64,i64,i64,i64
0,null,0,0,0,1,0,0,0
0,null,0,0,1,1,1,0,2
0,null,1,1,2,1,0,1,1
0,null,1,1,3,1,1,1,3
